# Tools in LangChain

In LangChain, a **tool** is an external function that an LLM can call to do something it cannot reliably do by itself. An LLM can generate text, but a tool lets it take actions or access external information.

## Simple Examples

| Tool          | What it does          |
| ------------- | ---------------------- |
| Search tool   | Search the web          |
| Calculator    | Perform calculations    |
| Weather API   | Get current weather     |
| SQL tool      | Query a database         |
| File tool     | Read files               |
| Email tool    | Send/read emails         |
| Python tool   | Execute Python            |
| API tool      | Call external APIs        |

In [1]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"It is sunny in { location}"

model_with_tools = model.bind_tools([get_weather])

In [8]:
response = model_with_tools.invoke("What is weather in ghaziabad")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "What is weather in ghaziabad". We need to get current weather. Use get_weather function with location "Ghaziabad".', 'tool_calls': [{'id': 'fc_e83e68f5-3904-4fb2-a352-103340415a5f', 'function': {'arguments': '{"location":"Ghaziabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 126, 'total_tokens': 186, 'completion_time': 0.127664174, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.048112516, 'prompt_tokens_details': None, 'queue_time': 0.320004212, 'total_time': 0.17577669}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3166198c1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04861-3ef7-77b3-af34-1385c1266efe-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Ghaziabad'}, 'id': 'fc_e83e68f5-3904-4fb2-a3

In [10]:
for tool_call in response.tool_calls:
    print(f"Tool: { tool_call['name']}")
    print(f"args: {tool_call['args']}")

Tool: get_weather
args: {'location': 'Ghaziabad'}


In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny.


In [12]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_c672c8ca-e67b-4f22-b074-bfaecf57b952', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 125, 'total_tokens': 165, 'completion_time': 0.084631189, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.026523908, 'prompt_tokens_details': None, 'queue_time': 0.29035152, 'total_time': 0.111155097}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04865-86b1-7c53-828f-098bbf5fb09c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_c672c8ca-e67b-4f22-b074-bfaecf57b9